# Building Your Own RAG System

## Assignment Overview

In this assignment, you'll build a complete Retrieval-Augmented Generation (RAG) system from scratch. You'll implement key components, compare different text representation strategies, and evaluate system performance.

**Due Date**: Monday, October 21, 11:59 PM

**Learning Objectives**:

- Understand how text representation quality affects downstream task performance
- Implement semantic search with vector embeddings
- Compare different embedding models and architectures
- Evaluate retrieval systems using appropriate metrics
- Build production-ready data pipelines

**Submission Requirements**:

- Completed Jupyter Notebook with all code cells executed
- Written responses to conceptual questions
- Performance comparison graphs
- Submit to: 'Code Practice Assignment 3 – Text Representation 2' under 'Assignments'

---

## Setup

In [ ]:
# Run this cell first
!pip install pinecone openai sentence-transformers datasets beautifulsoup4 requests

In [ ]:
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer, CrossEncoder
import hashlib
from datetime import datetime, timezone
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd

---

## Part A: Basic RAG Implementation

### Task A1: Implement Embedding Functions

Complete the following functions to generate embeddings:

In [ ]:
# TODO: Initialize your OpenAI client and Pinecone client
# HINT: Use userdata.get() or directly set API keys
# NOTE: Use the API keys provided during class
client = OpenAI(
    api_key='______'  # TODO: Fill in your OpenAI API key (use the key from class)
)

pc = Pinecone(
    api_key='______'  # TODO: Fill in your Pinecone API key (use the key from class)
)

# Configuration
INDEX_NAME = '______'  # TODO: Choose a name for your index
NAMESPACE = '______'   # TODO: Choose a namespace (e.g., 'default')
ENGINE = '______'      # TODO: Choose embedding model (e.g., 'text-embedding-3-small')

def get_embeddings(texts, engine=ENGINE):
    """
    Generate embeddings for a list of texts.

    Args:
        texts (list): List of strings to embed
        engine (str): OpenAI embedding model name

    Returns:
        list: List of embedding vectors
    """
    # TODO: Call the OpenAI API to create embeddings
    response = client.embeddings.create(
        input=______,  # TODO: What should be the input?
        model=______   # TODO: What model should we use?
    )

    # TODO: Extract embeddings from the response
    return [______ for d in list(response.data)]  # HINT: What attribute contains the embedding?

def get_embedding(text, engine=ENGINE):
    """
    Generate embedding for a single text.

    Args:
        text (str): Single string to embed
        engine (str): OpenAI embedding model name

    Returns:
        list: Embedding vector
    """
    # TODO: Use get_embeddings() to get embedding for a single text
    return ______[0]  # HINT: How do you get the first element?

# Test your implementation
test_embedding = get_embedding('hello world')
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

**Conceptual Question A1**: Why is it important to use the same embedding model for both indexing and querying? What would happen if you used different models? (Write 2-3 sentences)

**Answer A1**:

### Task A2: Implement Vector Database Setup
Complete the Pinecone index creation:

In [ ]:
def create_pinecone_index(index_name, dimension, metric='cosine'):
    """
    Create a Pinecone index if it doesn't exist.

    Args:
        index_name (str): Name of the index
        dimension (int): Vector dimensionality
        metric (str): Similarity metric ('cosine', 'euclidean', 'dotproduct')

    Returns:
        Index: Pinecone index object
    """
    # TODO: Check if index already exists
    if index_name not in ______:  # HINT: Use pc.list_indexes()
        print(f'Creating index {index_name}')
        pc.create_index(
            name=______,           # TODO: Fill in the index name
            dimension=______,      # TODO: Fill in the dimension
            metric=______,         # TODO: Fill in the metric
            spec=ServerlessSpec(
                cloud='______',    # TODO: Choose cloud provider ('aws', 'gcp', 'azure')
                region='______'    # TODO: Choose region (e.g., 'us-east-1')
            )
        )

    # TODO: Return the index object
    return pc.Index(name=______)

# TODO: Create your index
# HINT: What dimension should match your embedding model?
index = create_pinecone_index(INDEX_NAME, dimension=______, metric='cosine')

# Verify index was created
stats = index.describe_index_stats()
print(f"Index stats: {stats}")

**Conceptual Question A2**: Why did we choose cosine similarity as our metric? When would you use Euclidean distance instead? (Write 2-3 sentences)

**Answer A2**:

### Task A3: Implement Data Preparation Pipeline
Complete the data preparation functions:

In [ ]:
def my_hash(text):
    """
    Generate MD5 hash of text for use as document ID.

    Args:
        text (str): Input text

    Returns:
        str: Hexadecimal hash string
    """
    # TODO: Create and return MD5 hash
    return hashlib.md5(______.encode()).hexdigest()  # HINT: What should be hashed?

def prepare_for_pinecone(texts, engine=ENGINE, urls=None):
    """
    Prepare texts for Pinecone upload by generating embeddings and metadata.

    Args:
        texts (list): List of text strings
        engine (str): Embedding model name
        urls (list, optional): List of source URLs

    Returns:
        list: List of (id, embedding, metadata) tuples
    """
    # TODO: Get current timestamp
    now = datetime.now(timezone.utc).isoformat()

    # TODO: Generate embeddings for all texts
    embeddings = ______(texts, engine=engine)  # HINT: Which function generates embeddings?

    # TODO: Create list of tuples (id, embedding, metadata)
    responses = [
        (
            ______,                                    # TODO: Generate unique ID from text
            embedding,
            dict(text=text, date_uploaded=______)      # TODO: Add timestamp
        )
        for text, embedding in zip(texts, embeddings)
    ]

    # TODO: Add URLs to metadata if provided
    if urls and len(urls) == len(texts):
        for response, url in zip(responses, urls):
            response[-1]['______'] = url  # HINT: What key should store the URL?

    return responses

# Test your implementation
test_texts = ['Hello world', 'Text representation is important']
prepared = prepare_for_pinecone(test_texts)
print(f"Number of prepared items: {len(prepared)}")
print(f"First item ID: {prepared[0][0]}")
print(f"First item metadata: {prepared[0][2]}")

**Conceptual Question A3**: Why is deterministic hashing (MD5) useful for document IDs? What problems does it solve in a production RAG system? (Write 2-3 sentences)

**Answer A3**:

### Task A4: Implement Upload and Query Functions
Complete the upload and query functions:

In [ ]:
def upload_texts_to_pinecone(texts, namespace=NAMESPACE, batch_size=32,
                             show_progress_bar=True, urls=None):
    """
    Upload texts to Pinecone in batches.

    Args:
        texts (list): List of text strings
        namespace (str): Pinecone namespace
        batch_size (int): Batch size for processing
        show_progress_bar (bool): Show progress bar
        urls (list, optional): List of source URLs

    Returns:
        int: Total number of vectors uploaded
    """
    total_upserted = 0

    # TODO: Create range for batch processing
    _range = range(0, len(texts), ______)  # HINT: What is the step size?

    # Iterate through batches
    for i in tqdm(_range) if show_progress_bar else _range:
        # TODO: Extract current batch
        text_batch = texts[______:______]  # HINT: How do you slice from i to i+batch_size?

        # TODO: Prepare texts with or without URLs
        if urls:
            url_batch = urls[i: i + batch_size]
            prepared_texts = prepare_for_pinecone(______, urls=______)
        else:
            prepared_texts = prepare_for_pinecone(______)

        # TODO: Upload to Pinecone
        result = index.upsert(
            vectors=______,      # TODO: What should be uploaded?
            namespace=______     # TODO: Which namespace?
        )

        total_upserted += result['______']  # HINT: What key contains the count?

    return total_upserted

def query_from_pinecone(query, top_k=3, include_metadata=True):
    """
    Query Pinecone for similar documents.

    Args:
        query (str): Query text
        top_k (int): Number of results to return
        include_metadata (bool): Include metadata in results

    Returns:
        list: List of matching documents
    """
    # TODO: Get embedding for query
    query_embedding = ______(query, engine=ENGINE)  # HINT: Which function embeds single text?

    # TODO: Query Pinecone
    return index.query(
        vector=______,                      # TODO: What vector should be used?
        top_k=______,                       # TODO: How many results?
        namespace=______,                   # TODO: Which namespace?
        include_metadata=______             # TODO: Should metadata be included?
    ).get('matches')

# Test your implementation
sample_texts = [
    "The capital of France is Paris",
    "Python is a programming language",
    "Machine learning uses neural networks"
]

print("Uploading sample texts...")
count = upload_texts_to_pinecone(sample_texts, batch_size=2)
print(f"Uploaded {count} vectors")

print("\nQuerying...")
results = query_from_pinecone("What is the capital of France?", top_k=2)
for i, result in enumerate(results):
    print(f"\nResult {i+1}:")
    print(f"  Score: {result['score']:.4f}")
    print(f"  Text: {result['metadata']['text']}")

**Conceptual Question A4**: Why do we use batch processing instead of uploading texts one at a time? What are the trade-offs of different batch sizes? (Write 2-3 sentences)

**Answer A4**:

## Part B: Model Comparison and Evaluation
### Task B1: Implement Evaluation on MLQA Dataset
Load the dataset and implement evaluation:

In [ ]:
from datasets import load_dataset

# Load MLQA dataset
dataset = load_dataset("xtreme", "MLQA.en.en")
dataset['train'] = dataset['test']
dataset['test'] = dataset['validation']
del dataset['validation']

print(f"Test set size: {len(dataset['test'])}")
print(f"Sample question: {dataset['test'][0]['question']}")

# TODO: Index unique passages from test set
unique_passages = list(set(dataset['test']['context']))
print(f"Number of unique passages: {len(unique_passages)}")

# TODO: Upload passages to Pinecone in batches
print("Indexing passages...")
# HINT: Use batch_size=32 for efficiency
uploaded_count = upload_texts_to_pinecone(
    ______,              # TODO: What should be uploaded?
    batch_size=______,   # TODO: Choose batch size
    show_progress_bar=True
)
print(f"Uploaded {uploaded_count} passages")

# Create mapping from questions to correct passage hashes
q_to_hash = {data['question']: my_hash(data['context'])
             for data in dataset['test']}

def evaluate_retrieval(questions, correct_hashes, top_k=50):
    """
    Evaluate retrieval performance.

    Args:
        questions (list): List of questions
        correct_hashes (dict): Mapping from question to correct passage hash
        top_k (int): Number of results to retrieve

    Returns:
        dict: Evaluation results
    """
    results = []

    for question in tqdm(questions):
        # TODO: Query Pinecone
        retrieved = query_from_pinecone(______, top_k=______)

        # TODO: Find position of correct answer (if present)
        correct_position = None
        correct_hash = correct_hashes[question]

        for idx, result in enumerate(retrieved):
            if result['id'] == ______:  # TODO: Check if this is the correct passage
                correct_position = idx
                break

        results.append({
            'question': question,
            'correct_position': correct_position,
            'found': correct_position is not None
        })

    return results

# TODO: Evaluate on a sample of 100 questions
from random import Random
rng = Random(42)

sample_questions = list(set(dataset['test']['question']))
rng.shuffle(sample_questions)
sample_questions = sample_questions[:100]  # Use 100 questions for faster evaluation

print("Evaluating retrieval...")
eval_results = evaluate_retrieval(
    ______,        # TODO: What questions to evaluate?
    q_to_hash,
    top_k=50
)

# TODO: Calculate metrics
results_df = pd.DataFrame(eval_results)

# Calculate Recall@K
recall_at_k = {}
for k in [1, 3, 5, 10, 25, 50]:
    # TODO: Calculate what percentage of questions have correct answer in top-k
    recall = len(results_df[results_df['correct_position'] < ______]) / len(results_df)
    recall_at_k[k] = recall
    print(f"Recall@{k}: {recall:.3f}")

**Conceptual Question B1**: What does Recall@K measure? Why is Recall@1 particularly important for question answering systems? (Write 2-3 sentences)

**Answer B1**:

### Task B2: Compare Embedding Models
Implement comparison between OpenAI and open-source embeddings:

In [ ]:
# TODO: Load open-source embedding model
bi_encoder = SentenceTransformer("______")  # HINT: Try "sentence-transformers/all-mpnet-base-v2"

# TODO: Encode all test contexts with open-source model
print("Encoding contexts with open-source model...")
docs = dataset['test']['context']
doc_embeddings = bi_encoder.encode(
    ______,              # TODO: What to encode?
    batch_size=32,
    show_progress_bar=True
)

print(f"Open-source embedding dimension: {doc_embeddings.shape}")

from sentence_transformers.util import semantic_search

def find_most_similar(embedder, query, doc_embeddings, documents, k=50):
    """
    Find most similar documents using open-source embedder.

    Args:
        embedder: SentenceTransformer model
        query (str): Query text
        doc_embeddings: Pre-computed document embeddings
        documents (list): List of document texts
        k (int): Number of results

    Returns:
        list: List of (document, score, index) tuples
    """
    # TODO: Encode query
    query_embedding = embedder.encode([______], show_progress_bar=False)

    # TODO: Find similar documents
    similarities = semantic_search(______, ______, top_k=k)

    # TODO: Return results
    return [(documents[sim['corpus_id']], sim['score'], sim['corpus_id'])
            for sim in similarities[0]]

def evaluate_open_source(embedder, doc_embeddings, questions, correct_hashes, top_k=50):
    """
    Evaluate open-source embedding model.

    Args:
        embedder: SentenceTransformer model
        doc_embeddings: Pre-computed document embeddings
        questions (list): List of questions
        correct_hashes (dict): Mapping from question to correct passage hash
        top_k (int): Number of results to retrieve

    Returns:
        list: Evaluation results
    """
    results = []

    for question in tqdm(questions):
        # TODO: Find most similar documents
        retrieved = find_most_similar(
            ______,              # TODO: Which embedder?
            ______,              # TODO: What query?
            ______,              # TODO: Which embeddings?
            docs,
            k=top_k
        )

        # TODO: Find correct position
        correct_position = None
        correct_hash = correct_hashes[question]

        for idx, (passage, score, doc_idx) in enumerate(retrieved):
            if my_hash(passage) == ______:  # TODO: Check if correct
                correct_position = idx
                break

        results.append({
            'question': question,
            'correct_position': correct_position,
            'found': correct_position is not None
        })

    return results

# TODO: Evaluate open-source model
print("Evaluating open-source embeddings...")
os_eval_results = evaluate_open_source(
    bi_encoder,
    doc_embeddings,
    sample_questions,
    q_to_hash,
    top_k=50
)

# TODO: Compare results
os_results_df = pd.DataFrame(os_eval_results)

print("\n=== Performance Comparison ===")
print(f"\nOpenAI Embeddings (text-embedding-3-small):")
for k in [1, 5, 10]:
    recall = len(results_df[results_df['correct_position'] < k]) / len(results_df)
    print(f"  Recall@{k}: {recall:.3f}")

print(f"\nOpen-Source Embeddings (all-mpnet-base-v2):")
for k in [1, 5, 10]:
    recall = len(os_results_df[os_results_df['correct_position'] < k]) / len(os_results_df)
    print(f"  Recall@{k}: {recall:.3f}")

# TODO: Create comparison plot
plt.figure(figsize=(10, 6))

k_values = [1, 3, 5, 10, 25, 50]
openai_recalls = [len(results_df[results_df['correct_position'] < k]) / len(results_df)
                  for k in k_values]
os_recalls = [len(os_results_df[os_results_df['correct_position'] < k]) / len(os_results_df)
              for k in k_values]

plt.plot(k_values, openai_recalls, marker='o', label='______', linewidth=2)  # TODO: Add label
plt.plot(k_values, os_recalls, marker='s', label='______', linewidth=2)      # TODO: Add label

plt.xlabel('k (Number of Retrieved Documents)', fontsize=12)
plt.ylabel('Recall@k', fontsize=12)
plt.title('______', fontsize=14)  # TODO: Add title
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(k_values)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

**Conceptual Question B2**: Based on your results, what is the performance difference between OpenAI and open-source embeddings? What factors might explain this difference (e.g., model size, training data, dimensionality)? (Write 3-4 sentences)

**Answer B2**:

## Part C: Advanced Features
### Task C1: Implement Cross-Encoder Reranking
Add reranking to improve results:

In [ ]:
# TODO: Load pre-trained cross-encoder
cross_encoder = CrossEncoder('______', num_labels=1)  # HINT: Try 'cross-encoder/ms-marco-MiniLM-L-12-v2'

def rerank_results(query, retrieved_results, cross_encoder):
    """
    Rerank retrieved results using cross-encoder.

    Args:
        query (str): Query text
        retrieved_results (list): Initial retrieval results
        cross_encoder: CrossEncoder model

    Returns:
        list: Reranked results
    """
    if not retrieved_results:
        return []

    # TODO: Create query-document pairs
    sentence_pairs = [[______, result['metadata']['text']]
                      for result in retrieved_results]  # HINT: What should be paired with each document?

    # TODO: Score pairs with cross-encoder
    from torch import nn
    scores = cross_encoder.predict(______, activation_fct=nn.Sigmoid())

    # TODO: Sort by scores (descending)
    sorted_indices = np.argsort(scores)[::-1]  # Reverse for descending

    # TODO: Reorder results
    reranked = [retrieved_results[idx] for idx in sorted_indices]

    # Add cross-encoder scores
    for i, idx in enumerate(sorted_indices):
        reranked[i]['ce_score'] = scores[idx]

    return reranked

# TODO: Evaluate with reranking
def evaluate_with_reranking(questions, correct_hashes, top_k=50):
    """
    Evaluate retrieval with cross-encoder reranking.
    """
    results = []

    for question in tqdm(questions):
        # Stage 1: Retrieve candidates
        retrieved = query_from_pinecone(question, top_k=top_k)

        # TODO: Find position before reranking
        retrieved_position = None
        correct_hash = correct_hashes[question]
        for idx, result in enumerate(retrieved):
            if result['id'] == correct_hash:
                retrieved_position = idx
                break

        # Stage 2: Rerank
        reranked = rerank_results(______, ______, ______)  # TODO: Fill in arguments

        # TODO: Find position after reranking
        reranked_position = None
        for idx, result in enumerate(reranked):
            if result['id'] == correct_hash:
                reranked_position = idx
                break

        results.append({
            'question': question,
            'retrieved_position': retrieved_position,
            'reranked_position': reranked_position
        })

    return results

print("Evaluating with cross-encoder reranking...")
rerank_eval = evaluate_with_reranking(sample_questions, q_to_hash, top_k=50)
rerank_df = pd.DataFrame(rerank_eval)

# TODO: Compare with and without reranking
print("\n=== Reranking Impact ===")
for k in [1, 3, 5, 10]:
    before = len(rerank_df[rerank_df['retrieved_position'] < k]) / len(rerank_df)
    after = len(rerank_df[rerank_df['reranked_position'] < k]) / len(rerank_df)
    improvement = (after - before) * 100
    print(f"Recall@{k}:")
    print(f"  Before reranking: {before:.3f}")
    print(f"  After reranking:  {after:.3f}")
    print(f"  Improvement:      {improvement:+.1f}%")

**Conceptual Question C1**: Explain why cross-encoder reranking improves results compared to bi-encoder retrieval alone. What is the trade-off? (Write 2-3 sentences)

**Answer C1**:

**Submission Instructions:**
- Save your notebook as: `Assignment_YourName_YourStudentID.ipynb`
- **Submit to:** 'Code Practice Assignment 3 – Text Representation 2' section under 'Assignments'
- **Deadline:** Monday, October 21, 11:59 PM

**Example:**
- If your name is John Smith and student ID is 20241234:
  - `Assignment_MinjooSon_20251234.ipynb`